In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))

from tutorial._infra import torch_frontend as torch_nb
import snntorch as snn
torch_nb.print_setup()


beta = 0.85
num_steps = 10
num_inputs = 128
num_hidden = 64
num_outputs = 4
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc_in = nn.Linear(num_inputs, num_hidden)
        nn.init.uniform_(self.fc_in.weight, 0.0, 1.0)
        self.rlif  = snn.RLeaky(beta=beta, linear_features=num_hidden)
        self.fc_out = nn.Linear(num_hidden, num_outputs)
        self.lif_out = snn.Leaky(beta=beta)
 
    def forward(self, x):
        # x: [num_steps, batch, num_inputs]
        spk1, mem1 = self.rlif.init_rleaky()   # hidden spikes + membrane
        mem2 = self.lif_out.init_leaky()
 
        spk_rec, mem_rec = [], []
        for t in range(x.size(0)):
            cur1 = self.fc_in(x[t])
            spk1, mem1 = self.rlif(cur1, spk1, mem1)   # recurrence happens here
            cur2 = self.fc_out(spk1)
            spk2, mem2 = self.lif_out(cur2, mem2)
 
            spk_rec.append(spk2)
            mem_rec.append(mem2)
 
        return torch.stack(spk_rec), torch.stack(mem_rec)


In [ ]:
snn_module = Net().eval()
example_input = torch.randn(num_steps, 10, num_inputs)
out,out_mem = snn_module(example_input)

In [ ]:
print(out.shape)

In [ ]:
import torch
from torch_mlir import fx
from torch_mlir import fx
from torch_mlir.extras.fx_decomp_util import get_decomposition_table

snn_module = Net().eval()
example_input = torch.randn(num_steps, 10, num_inputs, dtype=torch.float32)

aten = torch.ops.aten

def rnn_tanh_cell_decomp(input, hx, w_ih, w_hh, b_ih=None, b_hh=None):
    igates = aten.mm(input, aten.t(w_ih))
    hgates = aten.mm(hx, aten.t(w_hh))
    if b_ih is not None:
        igates = aten.add(igates, b_ih)
    if b_hh is not None:
        hgates = aten.add(hgates, b_hh)
    return aten.tanh(aten.add(igates, hgates))

table = get_decomposition_table()          # torch-mlir's default table (a dict)
table[aten.rnn_tanh_cell.default] = rnn_tanh_cell_decomp
torch_module = fx.export_and_import(snn_module, example_input, func_name="kernel",decomposition_table=table)
torch_ir = torch_module.operation.get_asm()

torch_file = torch_nb.ARTIFACTS_DIR / "snn_rnn_net_torch.mlir"
torch_file.write_text(torch_ir)

print(f"Wrote Torch dialect IR → {torch_file.resolve()}")
print(torch_ir)

In [ ]:
from pathlib import Path

pipeline = (
    "builtin.module("
    "torch-function-to-torch-backend-pipeline,"
    "torch-backend-to-linalg-on-tensors-backend-pipeline,"
    "torch-verify-linalg-on-tensors-backend-contract,"
    "func.func(refback-generalize-tensor-concat)"
    ")"
)

torch_ir = torch_nb.ARTIFACTS_DIR / "snn_rnn_net_torch.mlir"
linalg_ir = torch_nb.ARTIFACTS_DIR / "snn_rnn_net_linalg.mlir"

torch_nb.run(
    [
        torch_nb.torch_mlir_opt,
        torch_ir,
        f"-pass-pipeline={pipeline}",
        "-o",
        linalg_ir,
    ]
)

print(f"Wrote Linalg IR → {linalg_ir}")
linalg_txt = linalg_ir.read_text()
print(linalg_txt)

In [ ]:
from tutorial._infra import cinm_frontend as cinm_nb

cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-cleanup-linalg,im2col-to-matmul)"
    ")"
)

snn_linalg_clean = torch_nb.ARTIFACTS_DIR / "snn_rnn_net_linalg_clean.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        linalg_ir,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_linalg_clean,
    ]
)

snn_linalg_clean_txt = snn_linalg_clean.read_text()
print(snn_linalg_clean_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(convert-linalg-to-cinm)"
    ")"
)

snn_cinm0 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm0.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_linalg_clean,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm0,
    ]
)

snn_cinm0_txt = snn_cinm0.read_text()
print(snn_cinm0_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemv tile-sizes=32x32},"
    "cinm-annotate-tiles{ops=gemm tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemv tile-sizes=32x32x32},"
    "cinm-annotate-tiles{ops=batch_gemm tile-sizes=32x32x32x32},"
    "cinm-annotate-tiles{ops=activate tile-sizes=32}"
    ")"
)

snn_cinm1 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm1.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm0,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm1,
    ]
)

snn_cinm1_txt = snn_cinm1.read_text()
print(snn_cinm1_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "cinm-gemm-to-gemv{split-dim=2}"
    ")"
)

snn_cinm2 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm2.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm1,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm2,
    ]
)

snn_cinm2_txt = snn_cinm2.read_text()
print(snn_cinm2_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "cinm-tiling"
    ")"
)

snn_cinm3 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm3.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm2,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm3,
    ]
)

snn_cinm3_txt = snn_cinm3.read_text()
print(snn_cinm3_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-decompose-accum)"
    ")"
)

snn_cinm4 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm4.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm3,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm4,
    ]
)

snn_cinm4_txt = snn_cinm4.read_text()
print(snn_cinm4_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(cinm-insert-quantization{ops=gemm,gemv qtype=i8 scale=0.03125 zp=0 rounding=nearest narrow-range=false})"
    ")"
)

snn_cinm5 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm5.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm4,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm5,
    ]
)

snn_cinm5_txt = snn_cinm5.read_text()
print(snn_cinm5_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "lower-affine,"
    "func.func(cinm-relower)"
    ")"
)

snn_cinm6 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm6.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm5,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm6,
    ]
)

snn_cinm6_txt = snn_cinm6.read_text()
print(snn_cinm6_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(linalg-generalize-named-ops,canonicalize,scf-for-loop-canonicalization),"
    "one-shot-bufferize{bufferize-function-boundaries=true function-boundary-type-conversion=identity-layout-map unknown-type-conversion=identity-layout-map}"
    ")"
)

snn_cinm7 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm7.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm6,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm7,
    ]
)

snn_cinm7_txt = snn_cinm7.read_text()
print(snn_cinm7_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "cinm-memory-cleanup,"
    "func.func(convert-cinm-to-cim,cim-mark-relower{ops=add})"
    ")"
)

snn_cinm8 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm8.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm7,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm8,
    ]
)

snn_cinm8_txt = snn_cinm8.read_text()
print(snn_cinm8_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(convert-cim-to-alpine,cim-cleanup-unsupported)"
    ")"
)

snn_cinm9 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm9.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm8,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm9,
    ]
)

snn_cinm9_txt = snn_cinm9.read_text()
print(snn_cinm9_txt)

In [ ]:
cinm_pipeline = (
    "builtin.module("
    "func.func(llvm-request-c-wrappers),"
    "convert-alpine-to-func,"
    "func.func(convert-linalg-to-loops),"
    "func.func(affine-expand-index-ops),"
    "lower-affine,"
    "convert-scf-to-cf,"
    "expand-strided-metadata,"
    "func.func(affine-expand-index-ops),"
    "lower-affine,"
    "convert-vector-to-llvm,"
    "convert-math-to-llvm,"
    "convert-arith-to-llvm,"
    "convert-index-to-llvm,"
    "convert-to-llvm,"
    "reconcile-unrealized-casts,"
    "canonicalize"
    ")"
)

snn_cinm10 = torch_nb.ARTIFACTS_DIR / "snn_rnn_cinm10.mlir"
cinm_nb.run(
    [
        cinm_nb.cinm_opt,
        snn_cinm9,
        f"-pass-pipeline={cinm_pipeline}",
        "-o",
        snn_cinm10,
    ]
)

snn_cinm10_txt = snn_cinm10.read_text()
print(snn_cinm10_txt)